# 2차 — EBM (Explainable Boosting Machine)

## 배경
지금까지 시도한 모든 모델(XGBoost, CatBoost, TabTransformer, lambdarank, rank:pairwise)이
LightGBM과 상관관계 0.96+로 너무 비슷했음. EBM은 **부스팅이지만 트리가 아니라 "피처 하나씩
순환적으로(cyclic gradient boosting)" 학습**하는 구조적으로 다른 알고리즘 — 디커럴레이션
가능성이 가장 높은 후보.

## 사전 확인 (2만 행 서브샘플)
- `interactions=0`: holdout AUC 0.72228, 28초
- `interactions=10`: holdout AUC 0.72487, 35초
→ interactions를 켜는 게 약간 더 나음. 전체 데이터(25만행)로는 fold당 5~7분, 5-fold 전체
**약 30분 예상** — 시간 넉넉히 잡고 실행해주세요.

## 설치
`uv add interpret-core` (의존성 충돌 적은 경량 버전. `interpret` 전체 패키지는 PyJWT 등과
충돌 가능성 있었음)

---
**저장 위치:** `experiment_history/2차/2차_ebm.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

N_SPLITS = 5
SEED = 1004  # 먼저 1개 시드로 확인, 괜찮으면 멀티시드로 확장
EBM_INTERACTIONS = 10


## 1. 데이터 로드 + 전처리 (LightGBM 노트북들과 동일)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))

y = df_train[TARGET_COL].values.astype(int)
X = df_train.drop(columns=[TARGET_COL])
print(f"train shape: {X.shape}")


train shape: (256351, 67)


## 2. EBM 5-fold CV (시간 꽤 걸림 — fold별 진행상황 출력)

In [4]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_ebm = np.zeros(len(y))

start = time.time()
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    t0 = time.time()
    ebm = ExplainableBoostingClassifier(
        random_state=SEED,
        n_jobs=-1,
        interactions=EBM_INTERACTIONS,
    )
    ebm.fit(X.iloc[tr_idx], y[tr_idx])
    oof_ebm[val_idx] = ebm.predict_proba(X.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y[val_idx], oof_ebm[val_idx])
    print(f"[fold {fold}] AUC={fold_auc:.5f}  ({time.time()-t0:.0f}s)")

print(f"\n총 소요 시간: {time.time()-start:.0f}s")
print(f"EBM 전체 OOF AUC: {roc_auc_score(y, oof_ebm):.5f}")
print()
print("비교 기준:")
print("  LightGBM 단일시드 baseline: 0.7400 ~ 0.74053")
print("  10시드 배깅 baseline: 0.74036 / 0.74037")


[fold 0] AUC=0.74060  (178s)
[fold 1] AUC=0.73759  (182s)
[fold 2] AUC=0.73603  (155s)
[fold 3] AUC=0.73630  (156s)
[fold 4] AUC=0.73792  (170s)

총 소요 시간: 842s
EBM 전체 OOF AUC: 0.73766

비교 기준:
  LightGBM 단일시드 baseline: 0.7400 ~ 0.74053
  10시드 배깅 baseline: 0.74036 / 0.74037


## 3. 기존 LightGBM OOF와 상관관계 (디커럴레이션 확인)

In [5]:
# 기존에 저장해둔 LightGBM 배깅 OOF가 있으면 비교 (없으면 이 셀 건너뛰어도 됨)
LGBM_OOF_PATH = SHARED_DIR / "blend_cache" / "oof_10seed_bagged.npy"
if LGBM_OOF_PATH.exists():
    oof_lgbm = np.load(LGBM_OOF_PATH)
    if len(oof_lgbm) == len(y):
        corr = np.corrcoef(oof_ebm, oof_lgbm)[0, 1]
        print(f"EBM vs LightGBM(10시드 배깅) OOF 상관관계: {corr:.4f}")
        print("참고: 지금까지 시도한 모델들(XGBoost/CatBoost/TabTransformer/rank-loss)은 전부 0.96+ 였음")
        print("이보다 낮으면 디커럴레이션 성공 — 블렌딩 가치 있음")
    else:
        print("길이가 안 맞아서 비교 불가")
else:
    print(f"{LGBM_OOF_PATH} 없음 — 직접 경로 확인해서 비교해보세요")

np.save(SHARED_DIR / "blend_cache" / "oof_ebm.npy", oof_ebm)
print("저장 완료: blend_cache/oof_ebm.npy")


EBM vs LightGBM(10시드 배깅) OOF 상관관계: 0.9869
참고: 지금까지 시도한 모델들(XGBoost/CatBoost/TabTransformer/rank-loss)은 전부 0.96+ 였음
이보다 낮으면 디커럴레이션 성공 — 블렌딩 가치 있음
저장 완료: blend_cache/oof_ebm.npy


## 4. 해석 가이드

- EBM 단독 OOF가 baseline(0.7400~)보다 낮은 건 정상 — 단독 성능보다 **상관관계가 낮은지**가 핵심
- 상관관계가 0.96보다 확실히 낮다면(예: 0.85~0.93), 처음으로 의미있게 디커럴레이션된 모델 →
  블렌딩 노트북(`2차_blend_comparison.ipynb`)에 추가해서 실제 블렌딩 효과 확인해볼 가치 있음
- 상관관계도 여전히 높다면, 이 데이터셋 자체가 어떤 알고리즘을 쓰든 비슷한 결론에 도달하는
  구조라는 추가 증거 (지금까지의 전체 패턴과 일치)
- 다음 단계로 `interactions=0`(더 빠름) vs `interactions=20+`(더 느림, 더 복잡)도 비교해볼 수 있음
